# 🚀 Product Description Generation Pipeline
## `google/flan-t5-base` · LoRA Fine-Tuning · FAISS RAG · T4 GPU

| # | Stage | Details |
|---|-------|---------|
| 1 | GPU Check | Verify T4, CUDA, VRAM |
| 2 | Install Dependencies | All required packages |
| 3 | Configuration | Hyper-parameters, paths |
| 4 | Dataset Loading | Synthetic (default) · HuggingFace · CSV upload |
| 5 | Cleaning & Preprocessing | Normalize, deduplicate, fill gaps |
| 6 | Augmentation | Scale to **2 000 – 5 000** rows |
| 7 | EDA Visualisation | Category distribution, description lengths |
| 8 | Train / Test Split | 90 / 10 stratified split |
| 9 | RAG Index | FAISS + Sentence-Transformers embeddings |
| 10 | LoRA Fine-Tuning | PEFT on flan-t5-base (seq2seq) |
| 11 | Hybrid Inference | Template baseline · Model-only · RAG-hybrid |
| 12 | Evaluation | ROUGE-1/2/L · BLEU |
| 13 | Results & Graphs | Bar charts, comparison tables, sample outputs |
| 14 | Export | CSV, FAISS index, adapter weights |

> ⚡ **Run cells strictly top → bottom.** Runtime → Change runtime type → **T4 GPU**.


In [1]:
# ── CELL 1 · GPU Verification ──────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "⚠ nvidia-smi not found")

import torch
cuda_ok = torch.cuda.is_available()
if cuda_ok:
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / (1024 ** 3)
    print(f"✅ GPU  : {props.name}")
    print(f"✅ VRAM : {vram:.1f} GB")
    if vram < 12:
        print("⚠ Warning: less than 12 GB VRAM detected — reduce batch_size if OOM occurs")
else:
    print("⚠ CUDA not available — training will run on CPU (very slow)")

print(f"✅ PyTorch : {torch.__version__}")
print(f"✅ Python  : {sys.version.split()[0]}")


Mon Apr 27 12:44:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ── CELL 2 · Dependency Installation ───────────────────────────────────────
# NOTE: bitsandbytes is intentionally NOT installed.
#   flan-t5-base (250 M params) fits in T4 VRAM in fp16 without quantisation.
#   bitsandbytes compiled without GPU support causes a triton.ops import crash
#   inside PEFT — removing it is the cleanest fix.

import subprocess, sys

def pip(*pkgs):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + list(pkgs)
    )

print("📦 Installing core ML stack …")
pip(
    "transformers==4.44.2",
    "peft==0.12.0",
    "accelerate==0.34.2",
    "datasets==2.21.0",
    "evaluate==0.4.3",
    "rouge-score==0.1.2",
    "sacrebleu==2.4.3",
)

print("📦 Installing embedding + search stack …")
pip(
    "sentence-transformers==3.1.1",
    "faiss-cpu==1.8.0.post1",
)

print("📦 Installing visualisation stack …")
pip("matplotlib>=3.8.0", "seaborn>=0.13.2", "tabulate>=0.9.0")

# ── Purge any pre-installed broken bitsandbytes from the Colab base image ────
print("🧹 Removing bitsandbytes (not needed, avoids triton.ops crash) …")
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "bitsandbytes", "-y", "-q"],
    capture_output=True,
)
# Clear any already-cached bitsandbytes/triton submodules in this session
import sys as _sys
_stale = [k for k in list(_sys.modules.keys())
          if "bitsandbytes" in k or ("triton" in k and "torch" not in k)]
for k in _stale:
    del _sys.modules[k]

# ── Verify critical imports ───────────────────────────────────────────────────
import importlib
required = [
    "transformers", "peft", "accelerate", "datasets",
    "evaluate", "rouge_score", "sacrebleu",
    "sentence_transformers", "faiss", "matplotlib", "seaborn"
]
for mod in required:
    ok = importlib.util.find_spec(mod.replace("-", "_")) is not None
    print(f"  {'✅' if ok else '❌'} {mod}")

print("\n✅ All dependencies ready!")


📦 Installing core ML stack …
📦 Installing embedding + search stack …
📦 Installing visualisation stack …
🧹 Removing bitsandbytes (not needed, avoids triton.ops crash) …
  ✅ transformers
  ✅ peft
  ✅ accelerate
  ✅ datasets
  ✅ evaluate
  ✅ rouge_score
  ✅ sacrebleu
  ✅ sentence_transformers
  ✅ faiss
  ✅ matplotlib
  ✅ seaborn

✅ All dependencies ready!


In [5]:
import subprocess, sys

result = subprocess.run([sys.executable, "-m", "pip", "install",
                "--force-reinstall", "--no-cache-dir",
                "numpy>=2.0", "pandas>=2.2",
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu128"],
               capture_output=False)
print("Return code:", result.returncode)

Return code: 1


In [6]:
# ── CELL 3 · Configuration — edit values here ──────────────────────────────
import os, random, re, json, hashlib, gc, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Data settings ─────────────────────────────────────────────────────────────
TARGET_SIZE   = 2000   # ← change to 5000 for a larger dataset (max recommended: 5000)
TRAIN_RATIO   = 0.90   # 90% train / 10% test

# ── Model settings ───────────────────────────────────────────────────────────
MODEL_NAME        = "google/flan-t5-base"   # seq2seq; reliable on T4 16 GB
MAX_INPUT_LENGTH  = 256
MAX_TARGET_LENGTH = 128

# ── LoRA hyper-parameters ────────────────────────────────────────────────────
LORA_R           = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05
LORA_TARGET_MODS = ["q", "v"]   # T5 attention projection layers

# ── Training hyper-parameters ────────────────────────────────────────────────
NUM_EPOCHS            = 3
BATCH_SIZE            = 8       # reduce to 4 if CUDA OOM
GRAD_ACCUMULATION     = 2
LEARNING_RATE         = 3e-4
WARMUP_STEPS          = 50
MAX_INFER_SAMPLES     = 200     # rows evaluated after training

# ── RAG settings ─────────────────────────────────────────────────────────────
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # 80 MB, fast
RAG_TOP_K       = 3

# ── Output directory ─────────────────────────────────────────────────────────
OUTPUT_DIR = Path("./pipeline_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {DEVICE}")
print(f"Model   : {MODEL_NAME}")
print(f"Dataset : {TARGET_SIZE} rows  (train={int(TARGET_SIZE*TRAIN_RATIO)}, test={TARGET_SIZE-int(TARGET_SIZE*TRAIN_RATIO)})")
print(f"Output  : {OUTPUT_DIR.resolve()}")


Device  : cuda
Model   : google/flan-t5-base
Dataset : 2000 rows  (train=1800, test=200)
Output  : /content/pipeline_outputs


In [23]:
# ── CELL 4 · Dataset Loading ─────────────────────────────────────────────────
import re, random
import pandas as pd
from pathlib import Path

SEED = 42

# ── User-configurable switches ────────────────────────────────────────────────
USE_HF        = True
HF_DATASET_ID = "amazon_polarity"
KAGGLE_SLUG   = ""

# ── Helper ────────────────────────────────────────────────────────────────────
def normalize(text):
    if not isinstance(text, str):
        text = str(text) if text is not None else ""
    return re.sub(r"\s+", " ", text).strip()

# ── A) Upload CSV via Colab dialog ────────────────────────────────────────────
from google.colab import files
print("📂 Please upload your CSV/Excel file …")
uploaded = files.upload()

raw_df = None
if uploaded:
    file_name = list(uploaded.keys())[0]
    try:
        raw_df = pd.read_csv(file_name) if file_name.endswith(".csv") else pd.read_excel(file_name)
        # Map your CSV columns → standard names
        raw_df = raw_df.rename(columns={
            "title":       "product_name",
            "description": "reference_description",
            # "category" already matches — no rename needed
        })
        raw_df = raw_df[["product_name", "category", "reference_description"]].dropna().reset_index(drop=True)
        print(f"✅ Loaded uploaded file: {len(raw_df)} rows from '{file_name}'")
    except Exception as e:
        print(f"⚠ Could not read uploaded file: {e}")

# ── B) HuggingFace free dataset (no API key) ──────────────────────────────────
if raw_df is None:
    print("⚠ No file uploaded — trying HuggingFace …")

try:
    from datasets import load_dataset
    print(f"⬇ Downloading HuggingFace dataset: {HF_DATASET_ID} …")
    hf_ds = load_dataset(HF_DATASET_ID, split="train")
    hf_df = hf_ds.to_pandas()
    if "title" in hf_df.columns and "content" in hf_df.columns:
        hf_df = hf_df.rename(columns={"title": "product_name", "content": "reference_description"})
        hf_df["category"] = "Product"
    if len(hf_df) > 20000:
        hf_df = hf_df.sample(20000, random_state=SEED)
    hf_df = hf_df[["product_name", "category", "reference_description"]].copy()
    print(f"✅ Loaded HF dataset: {len(hf_df)} rows")
except Exception as e:
    hf_df = pd.DataFrame(columns=["product_name", "category", "reference_description"])
    print(f"⚠ HuggingFace dataset unavailable: {e}")

# ── C) Synthetic fallback ─────────────────────────────────────────────────────
print("\n📝 Building synthetic seed data …")
SYNTHETIC_CATALOG = [
    ("Handwoven Cotton Saree","Apparel","Soft breathable cotton saree with a traditional handwoven pattern."),
    ("Copper Water Bottle (1L)","Kitchen","Pure copper bottle with leak-proof lid. BPA-free."),
    ("Yoga Mat with Carry Strap","Sports","Eco-friendly TPE yoga mat with 6 mm cushioning and alignment lines."),
    ("Wireless Bluetooth Earbuds","Electronics","True-wireless earbuds with 8-hour playback and IPX5 water resistance."),
    ("Wild Forest Honey (400 g)","Food","Raw unfiltered honey with no added sugar, rich in antioxidants."),
    # Add more rows as needed …
]
adjectives  = ["Premium","Handcrafted","Eco-Friendly","Organic","Traditional","Artisan","Durable","Compact"]
quality_adj = ["versatile","premium","handcrafted","lightweight","durable","elegant","natural"]
rng = random.Random(SEED)
templates = [
    "{adj} {name} designed for everyday reliability. {desc}",
    "A {quality} take on the classic {name}. {desc}",
    "Crafted with care, this {name} delivers outstanding {cat} value. {desc}",
]
synth_rows = []
for name, cat, desc in SYNTHETIC_CATALOG:
    synth_rows.append({"product_name": name, "category": cat, "reference_description": desc})
    for j in range(4):
        var_desc = normalize(rng.choice(templates).format(
            adj=rng.choice(adjectives), name=name, cat=cat.lower(),
            quality=rng.choice(quality_adj), desc=desc))[:500]
        synth_rows.append({"product_name": f"{rng.choice(adjectives)} {name} (Variant {j+1})",
                           "category": cat, "reference_description": var_desc})
synth_df = pd.DataFrame(synth_rows)
print(f"✅ Synthetic rows: {len(synth_df)}")

# ── Merge ALL three sources ───────────────────────────────────────────────────
frames = [df for df in [raw_df, hf_df, synth_df] if df is not None and len(df) > 0]
combined = pd.concat(frames, ignore_index=True)
print(f"\n📊 Combined raw rows: {len(combined)}")
combined.head(3)

📂 Please upload your CSV/Excel file …


Saving ecommerce_dataset_1000_converted.csv to ecommerce_dataset_1000_converted.csv
✅ Loaded uploaded file: 1000 rows from 'ecommerce_dataset_1000_converted.csv'
⬇ Downloading HuggingFace dataset: amazon_polarity …
✅ Loaded HF dataset: 20000 rows

📝 Building synthetic seed data …
✅ Synthetic rows: 25

📊 Combined raw rows: 21025


,product_name,category,reference_description
0,Men T-Shirt,Electronics,High quality men t-shirt with advanced feature...
1,Men T-Shirt,Clothing,High quality men t-shirt with advanced feature...
2,Mixer Grinder,Footwear,High quality mixer grinder with advanced featu...


In [24]:
import pandas as pd

# =========================
# ✅ 1. SELECT DATA SOURCE SAFELY
# =========================

if "final_df" in globals():
    df = final_df.copy()
    print("✅ Using final_df:", df.shape)

elif "df" in globals():
    df = df.copy()
    print("✅ Using existing df:", df.shape)

else:
    print("⚠️ No dataset found → creating fallback dataset")

    data = [
        {
            "product_name": "Wireless Bluetooth Earbuds",
            "category": "Electronics",
            "reference_description": "True wireless earbuds with noise cancellation, long battery life, and fast charging."
        },
        {
            "product_name": "Cotton Kurta",
            "category": "Apparel",
            "reference_description": "Comfortable cotton kurta with breathable fabric, ideal for daily wear."
        },
        {
            "product_name": "Yoga Mat",
            "category": "Sports",
            "reference_description": "Eco-friendly yoga mat with non-slip surface and cushioning support."
        }
    ]

    df = pd.DataFrame(data)
    print("✅ Fallback dataset created:", df.shape)

# =========================
# ✅ 2. ENSURE REQUIRED COLUMNS
# =========================

required_cols = ["product_name", "category", "reference_description"]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"❌ Missing required column: {col}")

# =========================
# ✅ 3. FILTER DATA (SAFE)
# =========================

df["reference_description"] = df["reference_description"].astype(str)

good_df = df[df["reference_description"].str.len() > 50].head(500).copy()

print("📊 After filter:", good_df.shape)

# =========================
# ✅ 4. CREATE infer_df
# =========================

infer_df = good_df[["product_name", "reference_description", "category"]].reset_index(drop=True)

print("\n✅ New infer_df:", infer_df.shape)

if len(infer_df) > 0:
    print("📌 Sample:", infer_df["product_name"].iloc[0])
else:
    print("⚠️ No rows passed filter. Try lowering threshold.")

✅ Using final_df: (2000, 4)
📊 After filter: (500, 4)

✅ New infer_df: (500, 3)
📌 Sample: Great History Unveiled


In [25]:
# ── CELL 5 · Data Cleaning & Preprocessing ─────────────────────────────────
NULL_TOKENS = {"nan", "none", "null", "na", "n/a", ""}

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Normalize all text columns
    for col in ["product_name", "category", "reference_description"]:
        df[col] = df[col].astype(str).map(normalize)

    # 2. Replace null-like strings
    for col in ["product_name", "category", "reference_description"]:
        df[col] = df[col].map(lambda x: "" if x.lower() in NULL_TOKENS else x)

    # 3. Drop rows with missing product names
    df = df[df["product_name"].str.strip().ne("")]

    # 4. Default category
    df["category"] = df["category"].replace("", "General")

    # 5. Fill missing descriptions
    mask = df["reference_description"].str.strip().eq("")
    df.loc[mask, "reference_description"] = (
        "A high-quality " + df.loc[mask, "category"].str.lower() +
        " product crafted for everyday use and lasting durability."
    )

    # 6. Truncate very long descriptions (keep first 500 chars)
    df["reference_description"] = df["reference_description"].str.slice(0, 500)

    # 7. Remove obvious duplicates
    before = len(df)
    df = df.drop_duplicates(subset=["product_name", "reference_description"], keep="first")
    print(f"  Deduplication: {before} → {len(df)} rows")

    # 8. Description length filter (too short = useless)
    df = df[df["reference_description"].str.split().str.len().ge(6)]

    df = df.reset_index(drop=True)
    df["is_augmented"] = 0
    return df

print("🧹 Cleaning data …")
cleaned_df = clean_df(combined)
print(f"  Rows after cleaning : {len(cleaned_df)}")
print(f"  Categories          : {cleaned_df['category'].nunique()}")
print(f"  Avg desc length     : {cleaned_df['reference_description'].str.split().str.len().mean():.1f} words")
print()
cleaned_df.info()
cleaned_df.head(5)


🧹 Cleaning data …
  Deduplication: 21023 → 20033 rows
  Rows after cleaning : 20033
  Categories          : 9
  Avg desc length     : 62.8 words

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20033 entries, 0 to 20032
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   product_name           20033 non-null  object
 1   category               20033 non-null  object
 2   reference_description  20033 non-null  object
 3   is_augmented           20033 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 626.2+ KB


,product_name,category,reference_description,is_augmented
0,Men T-Shirt,Electronics,High quality men t-shirt with advanced feature...,0
1,Mixer Grinder,Footwear,High quality mixer grinder with advanced featu...,0
2,Gaming Laptop,Footwear,High quality gaming laptop with advanced featu...,0
3,Running Shoes,Clothing,High quality running shoes with advanced featu...,0
4,Wireless Headphones,Clothing,High quality wireless headphones with advanced...,0


In [26]:
# ── CELL 6 · Data Augmentation → exactly TARGET_SIZE rows ──────────────────
# If data < TARGET_SIZE, synthetic variations are created.
# If data > TARGET_SIZE, a stratified sample is taken.

AUG_TEMPLATES = [
    "{name} is a {qual} {cat} product built for reliability. {base}",
    "Introducing the {name} — a {qual} addition to any {cat} collection. {base}",
    "The {name} combines {qual} design with practical {cat} functionality. {base}",
    "Experience quality with the {name}, a {qual} {cat} essential. {base}",
    "{name}: your {qual} {cat} companion for everyday use. {base}",
    "Crafted to last, the {name} redefines {qual} {cat} standards. {base}",
]
QUAL_TOKENS = ["versatile", "premium", "handcrafted", "lightweight", "durable",
               "elegant", "certified", "eco-conscious", "refined", "artisan"]

def augment_to_target(df: pd.DataFrame, target: int, seed: int) -> pd.DataFrame:
    rng_local = random.Random(seed)
    if len(df) >= target:
        return df.sample(target, random_state=seed).reset_index(drop=True)

    records  = df.to_dict("records")
    out      = list(records)
    while len(out) < target:
        src = dict(rng_local.choice(records))
        qual = rng_local.choice(QUAL_TOKENS)
        tmpl = rng_local.choice(AUG_TEMPLATES)
        new_desc = normalize(tmpl.format(
            name=src["product_name"],
            qual=qual,
            cat=(src["category"] or "product").lower(),
            base=src["reference_description"],
        ))[:500]
        out.append({
            "product_name":          src["product_name"] + f" (v{len(out)})",
            "category":              src["category"],
            "reference_description": new_desc,
            "is_augmented":          1,
        })
    result = pd.DataFrame(out[:target]).reset_index(drop=True)
    return result

print(f"📈 Augmenting to {TARGET_SIZE} rows …")
final_df = augment_to_target(cleaned_df, target=TARGET_SIZE, seed=SEED)

n_orig = int((final_df["is_augmented"] == 0).sum())
n_aug  = int((final_df["is_augmented"] == 1).sum())
print(f"  Original rows   : {n_orig}")
print(f"  Augmented rows  : {n_aug}")
print(f"  Total rows      : {len(final_df)}")

# Save processed dataset
processed_path = OUTPUT_DIR / "processed_dataset.csv"
final_df.to_csv(processed_path, index=False)
print(f"  Saved → {processed_path}")
final_df.head(3)


📈 Augmenting to 2000 rows …
  Original rows   : 2000
  Augmented rows  : 0
  Total rows      : 2000
  Saved → pipeline_outputs/processed_dataset.csv


,product_name,category,reference_description,is_augmented
0,i missed the boat,Product,I just did not get this one. The whole anti-es...,0
1,"Who said, he said, she said.",Product,"MY first exposure to Jesse Stone, Paradise, an...",0
2,A good general look at the history of western ...,Product,Wodehouse and Moffett do a good job of jamming...,0


In [28]:
# ── CELL 7 · Exploratory Data Analysis ──────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")          # safe for Colab — no display backend issues
import seaborn as sns
from pathlib import Path

# ── Guard: ensure required variables exist ────────────────────────────────────
if "final_df" not in dir() or final_df is None or len(final_df) == 0:
    raise RuntimeError("❌ 'final_df' is not defined or empty — run the earlier cells first.")

# ── Ensure required columns exist ────────────────────────────────────────────
for col in ["category", "reference_description", "is_augmented"]:
    if col not in final_df.columns:
        raise ValueError(f"❌ Column '{col}' missing from final_df. "
                         f"Available columns: {list(final_df.columns)}")

# ── Ensure OUTPUT_DIR exists ──────────────────────────────────────────────────
if "OUTPUT_DIR" not in dir():
    OUTPUT_DIR = Path("outputs")
OUTPUT_DIR = Path(OUTPUT_DIR)          # safe even if it's already a Path
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Cast is_augmented safely to int ──────────────────────────────────────────
final_df["is_augmented"] = final_df["is_augmented"].fillna(0).astype(int)

# ── Compute word counts (drop NaN descriptions first) ────────────────────────
desc_series = final_df["reference_description"].fillna("").astype(str)
word_counts  = desc_series.str.split().str.len()

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Dataset Overview", fontsize=15, fontweight="bold")

# Plot 1 · Category distribution
cat_counts = final_df["category"].value_counts().head(15)
axes[0].barh(
    cat_counts.index[::-1],
    cat_counts.values[::-1],
    color=sns.color_palette("Set2", len(cat_counts))
)
axes[0].set_title("Top Categories")
axes[0].set_xlabel("Count")

# Plot 2 · Description word-count distribution
axes[1].hist(word_counts, bins=30, color="#4C72B0", edgecolor="white", alpha=0.85)
axes[1].axvline(
    word_counts.mean(), color="red", linestyle="--",
    label=f"Mean = {word_counts.mean():.1f}"
)
axes[1].set_title("Description Word-Count Distribution")
axes[1].set_xlabel("Words")
axes[1].set_ylabel("Frequency")
axes[1].legend()

# Plot 3 · Original vs Augmented
n_original  = int((final_df["is_augmented"] == 0).sum())
n_augmented = int((final_df["is_augmented"] == 1).sum())

if n_original + n_augmented == 0:
    axes[2].text(0.5, 0.5, "No data", ha="center", va="center")
else:
    axes[2].pie(
        [n_original, n_augmented],
        labels=["Original", "Augmented"],
        autopct="%1.1f%%",
        colors=["#2ecc71", "#3498db"],
        startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 1.5}
    )
axes[2].set_title("Original vs Augmented Data Split")

plt.tight_layout()

out_path = OUTPUT_DIR / "eda_overview.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ EDA chart saved → {out_path}")

# ── Stats summary ─────────────────────────────────────────────────────────────
print(f"\nDataset stats:")
print(f"  Rows         : {len(final_df)}")
print(f"  Categories   : {final_df['category'].nunique()}")
print(f"  Avg words    : {word_counts.mean():.1f}")
print(f"  Min words    : {word_counts.min()}")
print(f"  Max words    : {word_counts.max()}")
print(f"  Original     : {n_original}")
print(f"  Augmented    : {n_augmented}")

✅ EDA chart saved → pipeline_outputs/eda_overview.png

Dataset stats:
  Rows         : 2000
  Categories   : 5
  Avg words    : 61.4
  Min words    : 12
  Max words    : 111
  Original     : 2000
  Augmented    : 0


In [29]:
# ── CELL 8 · Train / Test Split ─────────────────────────────────────────────
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    final_df,
    train_size=TRAIN_RATIO,
    shuffle=True,
    random_state=SEED
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

train_df.to_csv(OUTPUT_DIR / "train_split.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test_split.csv",  index=False)

print(f"✅ Train rows : {len(train_df)}")
print(f"✅ Test rows  : {len(test_df)}")

# Create input→output pairs for the model
# Input  : "Generate a product description. Product: X Category: Y"
# Output : reference description

def make_pairs(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in df.iterrows():
        prompt   = (f"Generate a concise product description.\n"
                    f"Product: {normalize(r['product_name'])}\n"
                    f"Category: {normalize(r['category'])}\n"
                    f"Description:")
        response = normalize(r["reference_description"])
        rows.append({"prompt": prompt, "response": response})
    return pd.DataFrame(rows)

train_pairs = make_pairs(train_df)
test_pairs  = make_pairs(test_df)

print(f"\nSample input:\n{train_pairs['prompt'].iloc[0]}")
print(f"\nSample output:\n{train_pairs['response'].iloc[0]}")


✅ Train rows : 1800
✅ Test rows  : 200

Sample input:
Generate a concise product description.
Product: Boring - lost interest
Category: Product
Description:

Sample output:
This book is snippets of conversations from people who were influences in the .com era. Unfortunately, there are so many people, I have no idea who JOHN is or what project he was related with. And I don't care. The book does nothing to tell a story. Its not really a book...its more like journal someone would use to write a book.I am very interested in the real-life stories of .com businesses...how they got started, how big they got and how they fell from grace.This is not one of those books.


In [35]:
# ── CELL 9 · Tokenization (FIXED) ───────────────────────────────────────────
from datasets import Dataset # Import the Dataset class

def tokenize_function(examples):
    # ── Step 1: Tokenize inputs — NO padding here, collator handles it ────────
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_INPUT_LENGTH,
        padding=False,          # ← CRITICAL: never pad here
        truncation=True,
    )

    # ── Step 2: Tokenize labels via text_target ───────────────────────────────
    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_TARGET_LENGTH,
        padding=False,          # ← CRITICAL: never pad here
        truncation=True,
    )

    # ── Step 3: Assign label ids — DO NOT replace anything with -100 here ─────
    # DataCollatorForSeq2Seq will replace pad tokens → -100 automatically
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

# Convert pandas DataFrames to HuggingFace Datasets and rename columns
# (train_pairs and test_pairs are created in Cell 8)
train_dataset = Dataset.from_pandas(train_pairs).rename_columns({
    "prompt": "input_text",
    "response": "target_text"
})
test_dataset = Dataset.from_pandas(test_pairs).rename_columns({
    "prompt": "input_text",
    "response": "target_text"
})

# ── Tokenize datasets ─────────────────────────────────────────────────────────
tok_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names, # Remove original columns like 'input_text', 'target_text' after tokenization
    desc="Tokenising train",
)
tok_test = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=test_dataset.column_names,
    desc="Tokenising test",
)

print(f"✅ tok_train : {len(tok_train)} rows")
print(f"✅ tok_test  : {len(tok_test)} rows")

# ── Quick sanity check — labels must NOT be all -100 ─────────────────────────
sample_labels = tok_train[0]["labels"]
non_masked    = [t for t in sample_labels if t != -100]
assert len(non_masked) > 0, "❌ All labels are -100 — model cannot learn!"
print(f"✅ Label check passed  : {len(non_masked)} / {len(sample_labels)} tokens are real")
print(f"   First 10 label ids : {sample_labels[:10]}")
print(f"   Decoded target     : {tokenizer.decode(non_masked, skip_special_tokens=True)}")

Tokenising train:   0%|          | 0/1800 [00:00<?, ? examples/s]

Tokenising test:   0%|          | 0/200 [00:00<?, ? examples/s]

✅ tok_train : 1800 rows
✅ tok_test  : 200 rows
✅ Label check passed  : 127 / 127 tokens are real
   First 10 label ids : [100, 484, 19, 3, 20317, 4995, 7, 13, 9029, 45]
   Decoded target     : This book is snippets of conversations from people who were influences in the.com era. Unfortunately, there are so many people, I have no idea who JOHN is or what project he was related with. And I don't care. The book does nothing to tell a story. Its not really a book...its more like journal someone would use to write a book.I am very interested in the real-life stories of.com businesses...how they got started, how big they got and how they fell from grace.This is not one of those books.


In [36]:
# ── CELL 10 · Data Collator (FIXED) ─────────────────────────────────────────
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,        # ← pads labels with -100, not pad_token_id
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
    return_tensors="pt",
)

# ── Batch sanity check ────────────────────────────────────────────────────────
sample_batch  = data_collator([tok_train[i] for i in range(4)])
label_batch   = sample_batch["labels"]
non_neg       = (label_batch != -100).sum().item()
total         = label_batch.numel()

assert non_neg > 0, "❌ Entire label batch is -100 after collation!"
print(f"✅ Collator check passed")
print(f"   Batch label shape  : {label_batch.shape}")
print(f"   Real tokens        : {non_neg} / {total}  ({100*non_neg/total:.1f}%)")
print(f"   Input IDs shape    : {sample_batch['input_ids'].shape}")

✅ Collator check passed
   Batch label shape  : torch.Size([4, 128])
   Real tokens        : 330 / 512  (64.5%)
   Input IDs shape    : torch.Size([4, 32])


In [ ]:
!pip install -q "numpy==1.26.4" faiss-cpu pandas torch torchvision --force-reinstall

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
datasets 2.21.0 requires fsspec[http]<=2024.6.1,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
libraft-cu12 26.2.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.11.0 which is incompatible.
libcuvs-cu12 26

In [37]:
# ── CELL 9 · RAG Index (FAISS + Sentence-Transformers) ─────────────────────
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

print(f"⬇ Loading embedding model: {EMBEDDING_MODEL} …")
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

# Documents = "Product | Category | Description"
rag_documents = (
    "Product: " + final_df["product_name"].astype(str) +
    " | Category: " + final_df["category"].astype(str) +
    " | Description: " + final_df["reference_description"].astype(str)
).tolist()

print(f"🔢 Encoding {len(rag_documents)} documents …")
rag_vectors = embedder.encode(
    rag_documents,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=128,
    normalize_embeddings=True,
).astype(np.float32)

# Build FAISS IndexFlatIP (cosine similarity via normalised vectors)
index = faiss.IndexFlatIP(rag_vectors.shape[1])
index.add(rag_vectors)
print(f"✅ FAISS index built: {index.ntotal} vectors, dim={rag_vectors.shape[1]}")

faiss_path = OUTPUT_DIR / "product_rag.faiss"
meta_path  = OUTPUT_DIR / "product_rag_meta.jsonl"
faiss.write_index(index, str(faiss_path))

with meta_path.open("w", encoding="utf-8") as f:
    for i, row in final_df.iterrows():
        f.write(json.dumps({
            "id": i,
            "product_name":          row["product_name"],
            "category":              row["category"],
            "reference_description": row["reference_description"],
        }, ensure_ascii=False) + "\n")

print(f"  Saved FAISS index → {faiss_path}")
print(f"  Saved metadata    → {meta_path}")

def retrieve(query: str, top_k: int = RAG_TOP_K) -> list[str]:
    qvec = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    _, idxs = index.search(qvec, top_k)
    return [rag_documents[i] for i in idxs[0] if 0 <= i < len(rag_documents)]

# Smoke test
test_q = "Copper water bottle kitchen"
print(f"\nRetrieval smoke-test for '{test_q}':")
for hit in retrieve(test_q):
    print(f"  • {hit[:90]} …")


⬇ Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 …
🔢 Encoding 2000 documents …


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

✅ FAISS index built: 2000 vectors, dim=384
  Saved FAISS index → pipeline_outputs/product_rag.faiss
  Saved metadata    → pipeline_outputs/product_rag_meta.jsonl

Retrieval smoke-test for 'Copper water bottle kitchen':
  • Product: Quit working within a year | Category: Product | Description: This is an easy rev …
  • Product: Waste of money | Category: Product | Description: I liked this machine at first b …
  • Product: Tastes great! | Category: Product | Description: I was worried that this wouldn't …


In [38]:
# ── CELL 10 · Load Model + Apply LoRA ──────────────────────────────────────

# ── Guard: purge any bitsandbytes still cached in this Python session ─────────
import sys as _sys
_stale = [k for k in list(_sys.modules.keys())
          if "bitsandbytes" in k or ("triton" in k and "torch" not in k)]
if _stale:
    print(f"  Removing stale cached modules: {_stale}")
    for k in _stale:
        del _sys.modules[k]

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, TaskType, get_peft_model

# ── Tokenizer ─────────────────────────────────────────────────────────────────
print(f"⬇ Loading tokenizer: {MODEL_NAME} …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# ── Model — fp16 on GPU, fp32 on CPU ─────────────────────────────────────────
print(f"⬇ Loading model: {MODEL_NAME} …")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

# ── LoRA configuration ────────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODS,   # ["q", "v"] — T5 attention projections
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Tokenisation helper ───────────────────────────────────────────────────────
# FIX: use tokenizer(text_target=...) — as_target_tokenizer() is deprecated
#      in transformers ≥ 4.33 and removed in some builds.
import pandas as pd
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq

def tokenize_pairs(pairs_df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(pairs_df, preserve_index=False)

    def _tok(batch):
        # Encode inputs
        model_inputs = tokenizer(
            batch["prompt"],
            max_length=MAX_INPUT_LENGTH,
            truncation=True,
            padding="max_length",
        )
        # Encode targets using text_target= (no deprecated context manager)
        label_enc = tokenizer(
            text_target=batch["response"],
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            padding="max_length",
        )
        # Mask padding positions so they are ignored in the loss
        model_inputs["labels"] = [
            [-100 if tok_id == tokenizer.pad_token_id else tok_id for tok_id in ids]
            for ids in label_enc["input_ids"]
        ]
        return model_inputs

    ds = ds.map(_tok, batched=True, remove_columns=["prompt", "response"])
    ds.set_format(type="torch")
    return ds

print("\n🔢 Tokenising train and test pairs …")
tok_train = tokenize_pairs(train_pairs)
tok_test  = tokenize_pairs(test_pairs)
print(f"  Train examples : {len(tok_train)}")
print(f"  Test examples  : {len(tok_test)}")

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)
print("✅ Model and tokeniser ready")


⬇ Loading tokenizer: google/flan-t5-base …
⬇ Loading model: google/flan-t5-base …
trainable params: 1,769,472 || all params: 249,347,328 || trainable%: 0.7096

🔢 Tokenising train and test pairs …


Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

  Train examples : 1800
  Test examples  : 200
✅ Model and tokeniser ready


In [39]:
# ── CELL 11 · LoRA Fine-Tuning (FIXED) ──────────────────────────────────────
import json, inspect
import numpy as np
import torch
import matplotlib.pyplot as plt
import evaluate
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Seq2SeqTrainer returns (sequences, scores) — take only sequences
    if isinstance(preds, tuple):
        preds = preds[0]

    # Clip out-of-vocab ids before decoding
    preds   = np.clip(preds,  0, tokenizer.vocab_size - 1)
    labels  = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels  = np.clip(labels, 0, tokenizer.vocab_size - 1)

    decoded_preds  = tokenizer.batch_decode(preds,   skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels,  skip_special_tokens=True)

    # Strip whitespace
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )
    result["gen_len"] = np.mean([len(p.split()) for p in decoded_preds])
    return {k: round(v, 4) for k, v in result.items()}

# ── Training arguments ────────────────────────────────────────────────────────
training_output_dir = str(OUTPUT_DIR / "training_checkpoints")

train_kwargs = dict(
    output_dir=training_output_dir,
    overwrite_output_dir=True,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    bf16=False,
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rouge2",     # ← changed from eval_loss → ROUGE
    greater_is_better=True,             # ← higher ROUGE = better
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,             # ← beam search for better ROUGE
    report_to=[],
    seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

sig = inspect.signature(Seq2SeqTrainingArguments.__init__)
train_kwargs["eval_strategy" if "eval_strategy" in sig.parameters else "evaluation_strategy"] = "epoch"

training_args = Seq2SeqTrainingArguments(**train_kwargs)

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_test,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,    # ← added ROUGE callback
)

print("🏋 Starting fine-tuning …")
print(f"   Epochs      : {NUM_EPOCHS}")
print(f"   Batch size  : {BATCH_SIZE}")
print(f"   LR          : {LEARNING_RATE}")

try:
    train_result = trainer.train()
    print("\n✅ Training complete!")
    print(json.dumps({k: round(v, 4) for k, v in train_result.metrics.items()}, indent=2))
except torch.cuda.OutOfMemoryError:
    print("⚠ CUDA OOM — reduce BATCH_SIZE and re-run from Cell 10")
    raise

# ── Save adapter ──────────────────────────────────────────────────────────────
adapter_dir = OUTPUT_DIR / "lora_adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"💾 LoRA adapter saved → {adapter_dir}")

# ── Plot loss curves ──────────────────────────────────────────────────────────
log_history  = trainer.state.log_history
train_losses = [(e["epoch"], e["loss"])      for e in log_history if "loss"      in e and "eval_loss" not in e]
eval_losses  = [(e["epoch"], e["eval_loss"]) for e in log_history if "eval_loss" in e]
rouge2_scores= [(e["epoch"], e["eval_rouge2"]) for e in log_history if "eval_rouge2" in e]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

if train_losses:
    ax1.plot(*zip(*train_losses), "o-",  label="Train Loss", color="#2980b9")
if eval_losses:
    ax1.plot(*zip(*eval_losses),  "s--", label="Eval Loss",  color="#e74c3c")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Training & Evaluation Loss")
ax1.legend(); ax1.grid(alpha=0.3)

if rouge2_scores:
    ax2.plot(*zip(*rouge2_scores), "D-", label="ROUGE-2", color="#27ae60")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("ROUGE-2")
ax2.set_title("Validation ROUGE-2 Score")
ax2.axhline(0.4, color="orange", linestyle="--", label="Target (0.4)")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_loss.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Loss + ROUGE curves saved")

🏋 Starting fine-tuning …
   Epochs      : 3
   Batch size  : 8
   LR          : 0.0003


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
0,0.000000,3.552746,0.092000,0.024500,0.079100,0.079200,29.900000
2,0.000000,3.552746,0.092000,0.024500,0.079100,0.079200,29.900000



✅ Training complete!
{
  "train_runtime": 629.5704,
  "train_samples_per_second": 8.577,
  "train_steps_per_second": 0.534,
  "total_flos": 1855239242121216.0,
  "train_loss": 0.0,
  "epoch": 2.9867
}
💾 LoRA adapter saved → pipeline_outputs/lora_adapter
✅ Loss + ROUGE curves saved


In [40]:
# ── CELL 12 · Hybrid Inference — Template · Model-only · RAG-hybrid ─────────
# This cell defines the inference helpers and runs all three prediction modes
# against MAX_INFER_SAMPLES rows from the test set.

import torch
from tqdm.auto import tqdm
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# ── 1. Inference subset ──────────────────────────────────────────────────────
infer_df = test_df.sample(
    min(MAX_INFER_SAMPLES, len(test_df)), random_state=SEED
).reset_index(drop=True)
print(f"📊 Inference rows : {len(infer_df)}")

# ── 2. Template fallback helper ──────────────────────────────────────────────
def template_fallback(name: str, category: str, ref: str = "") -> str:
    """Rule-based description when the model is unavailable or produces empty output."""
    base = ref.strip() if ref.strip() else (
        f"A high-quality {category.lower()} product designed for everyday use."
    )
    return f"{name}: {base}"

# ── 3. Model generate helper ─────────────────────────────────────────────────
model.eval()

def generate(prompt: str, max_new_tokens: int = MAX_TARGET_LENGTH) -> str:
    """Run seq2seq inference for a single prompt string."""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    ).to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

# ── 4. Rebuild RAG index on inference subset descriptions ────────────────────
#    (uses the global embedder + index already built in Cell 9)
print("\n🔎 RAG index ready (from Cell 9)")

# ── 5. Run all three inference modes ─────────────────────────────────────────
preds = {"template_baseline": [], "model_only": [], "hybrid_rag": []}
contexts_col = []
errors = []

for i, (_, row) in enumerate(tqdm(infer_df.iterrows(), total=len(infer_df), desc="Inference")):
    name  = normalize(str(row["product_name"]))
    cat   = normalize(str(row["category"]))
    ref   = normalize(str(row["reference_description"]))
    query = f"{name} {cat}"

    # Retrieve RAG context
    try:
        ctx = retrieve(query, top_k=RAG_TOP_K)
    except Exception as e:
        ctx = []
        errors.append(f"row {i} retrieve: {e}")
    contexts_col.append(" || ".join(ctx))

    # Mode 1 — Template baseline
    preds["template_baseline"].append(template_fallback(name, cat, ref))

    # Mode 2 — Model only
    try:
        p2 = generate(
            f"Generate a concise product description.\nProduct: {name}\nCategory: {cat}\nDescription:"
        )
        preds["model_only"].append(p2 if p2 else template_fallback(name, cat, ref))
    except Exception as e:
        errors.append(f"row {i} model_only: {e}")
        preds["model_only"].append(template_fallback(name, cat, ref))

    # Mode 3 — Hybrid RAG
    ctx_str = "\n".join(f"- {c[:120]}" for c in ctx[:3])
    hybrid_prompt = (
        f"Write a product description for this item.\n"
        f"Product: {name}\nCategory: {cat}\n"
        f"Key details: {ref[:200]}\n"
        f"Additional context:\n{ctx_str}\nDescription:"
    )
    try:
        p3 = generate(hybrid_prompt)
        preds["hybrid_rag"].append(p3 if p3 else template_fallback(name, cat, ref))
    except Exception as e:
        errors.append(f"row {i} hybrid_rag: {e}")
        preds["hybrid_rag"].append(template_fallback(name, cat, ref))

# ── 6. Attach predictions to dataframe ──────────────────────────────────────
for mode, ps in preds.items():
    infer_df[f"pred_{mode}"] = ps
infer_df["retrieved_context"] = contexts_col

if errors:
    print(f"\n⚠ {len(errors)} inference errors (showing first 5):")
    for e in errors[:5]:
        print(f"  {e}")

print(f"\n✅ Inference complete. Columns: {infer_df.columns.tolist()}")
print({k: len(v) for k, v in preds.items()})

# Sample outputs
print("\n── Sample predictions ────────────────────────────────────────────")
row0 = infer_df.iloc[0]
print(f"Product   : {row0['product_name']}")
print(f"Reference : {row0['reference_description'][:120]}")
print(f"Template  : {row0['pred_template_baseline'][:120]}")
print(f"Model     : {row0['pred_model_only'][:120]}")
print(f"RAG       : {row0['pred_hybrid_rag'][:120]}")

📊 Inference rows : 200

🔎 RAG index ready (from Cell 9)


Inference:   0%|          | 0/200 [00:00<?, ?it/s]


✅ Inference complete. Columns: ['product_name', 'category', 'reference_description', 'is_augmented', 'pred_template_baseline', 'pred_model_only', 'pred_hybrid_rag', 'retrieved_context']
{'template_baseline': 200, 'model_only': 200, 'hybrid_rag': 200}

── Sample predictions ────────────────────────────────────────────
Product   : The most usefull and powerfull book for 3dsmax ever
Reference : since the dawn of maxscript there has little of no help for its aplication thus resulting in its use rating less than a 
Template  : The most usefull and powerfull book for 3dsmax ever: since the dawn of maxscript there has little of no help for its apl
Model     : The most usefull and powerfull book for 3dsmax ever
RAG       : Very useful book for interpreter training


In [50]:
# ── CELL A · Data Pipeline ───────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
from typing import Any

# ── Globals with safe defaults ────────────────────────────────────────────────
SEED              = 42
MAX_INFER_SAMPLES = 200

# ── normalize must be defined before use ─────────────────────────────────────
def normalize(text: Any) -> str:
    if not isinstance(text, str):
        text = str(text) if text is not None else ""
    return re.sub(r"\s+", " ", text).strip()

# ── 1. Ensure df exists ───────────────────────────────────────────────────────
try:
    df
    print(f"✅ df already in memory: {df.shape}")
except NameError:
    print("⚠ df not found — loading from CSV …")
    df = pd.read_csv("your_dataset.csv")
    print(f"✅ df loaded: {df.shape}")

print("\n📌 Available columns:", df.columns.tolist())

# ── 2. Auto-map columns ───────────────────────────────────────────────────────
# ✅ FIX 1: removed 'features' and 'details' — they don't exist in this dataset
col_map = {"title": None, "main_category": None}

for col in df.columns:
    c = col.lower()
    if col_map["title"]         is None and ("title" in c or "name"     in c): col_map["title"]         = col
    if col_map["main_category"] is None and  "category" in c:                  col_map["main_category"] = col

print("✅ Column mapping:", col_map)

# ── 3. Fill any unmapped columns with empty ───────────────────────────────────
for key in col_map:
    if col_map[key] is None:
        print(f"⚠ No column found for '{key}' → creating empty column")
        df[key] = ""
        col_map[key] = key

# ── 4. Use reference_description directly ────────────────────────────────────
# ✅ FIX 2: removed safe_parse + extract_reference — reference_description
#           already exists and is good. Rebuilding it from title alone
#           (with empty features/details) was overwriting it with short strings,
#           causing almost all rows to fail the length filter.
df["reference_description"] = df["reference_description"].fillna("").apply(normalize)

# ── 5. Filter + build infer_df ────────────────────────────────────────────────
# ✅ FIX 3: lowered threshold 100 → 30 to be robust to shorter descriptions
good_df = df[df["reference_description"].str.len() > 30].copy()
good_df = good_df.sample(min(MAX_INFER_SAMPLES, len(good_df)),
                          random_state=SEED).reset_index(drop=True)

infer_df = good_df[[
    col_map["title"],
    "reference_description",
    col_map["main_category"]
]].rename(columns={
    col_map["title"]:         "product_name",
    col_map["main_category"]: "category"
}).reset_index(drop=True)

infer_df["product_name"] = infer_df["product_name"].fillna("Unknown Product").apply(normalize)
infer_df["category"]     = infer_df["category"].fillna("General").apply(normalize)
infer_df["reference_description"] = infer_df["reference_description"].fillna("").apply(normalize)

print(f"\n✅ infer_df shape : {infer_df.shape}")
print(f"   Columns        : {infer_df.columns.tolist()}")
if len(infer_df) == 0:
    raise ValueError("❌ infer_df is empty — lower the >30 char filter or check column mapping")

print(f"\n📌 Sample row:")
print(f"   product_name : {infer_df['product_name'].iloc[0]}")
print(f"   category     : {infer_df['category'].iloc[0]}")
print(f"   description  : {infer_df['reference_description'].iloc[0][:100]} …")

✅ df already in memory: (2000, 6)

📌 Available columns: ['product_name', 'category', 'reference_description', 'is_augmented', 'features', 'details']
✅ Column mapping: {'title': 'product_name', 'main_category': 'category'}

✅ infer_df shape : (200, 3)
   Columns        : ['product_name', 'reference_description', 'category']

📌 Sample row:
   product_name : Traditional Badminton Racket (Graphite) (Variant 2)
   category     : Sports
   description  : Traditional Badminton Racket (Graphite) (Variant 2) …


In [51]:
# ── CELL B · FAISS RAG Index ─────────────────────────────────────────────────
import faiss
import numpy as np
from typing import List
from sentence_transformers import SentenceTransformer

# ── Guard ─────────────────────────────────────────────────────────────────────
if "infer_df" not in dir() or len(infer_df) == 0:
    raise RuntimeError("❌ infer_df missing — run Cell A first")

RAG_TOP_K = 3   # number of retrieved docs per query

# ── Build corpus ──────────────────────────────────────────────────────────────
rag_docs: List[str] = infer_df["reference_description"].tolist()

# ── Encode ────────────────────────────────────────────────────────────────────
print("⬇ Loading SentenceTransformer …")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print(f"🔢 Encoding {len(rag_docs)} documents …")
doc_embeddings = embedder.encode(
    rag_docs,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # ← handles L2 norm internally; no manual step needed
)

# ── Build FAISS index (cosine via inner product on unit vectors) ───────────────
dim   = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings.astype(np.float32))
print(f"✅ FAISS index built  : {index.ntotal} docs  |  dim={dim}")

# ── Retrieve function ─────────────────────────────────────────────────────────
def retrieve(query: str, top_k: int = RAG_TOP_K) -> List[str]:
    """Return top_k most similar descriptions for a query string."""
    q_emb = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    scores, ids = index.search(q_emb, top_k)
    return [rag_docs[i] for i in ids[0] if 0 <= i < len(rag_docs)]

# ── Quick test ────────────────────────────────────────────────────────────────
test_results = retrieve(infer_df["product_name"].iloc[0])
print(f"\n🔍 Test retrieve → {len(test_results)} results")
print(f"   Top match : {test_results[0][:100]} …")

⬇ Loading SentenceTransformer …
🔢 Encoding 200 documents …


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

✅ FAISS index built  : 200 docs  |  dim=384

🔍 Test retrieve → 3 results
   Top match : Traditional Badminton Racket (Graphite) (Variant 2) …


In [52]:
# ── CELL C · Inference Loop ──────────────────────────────────────────────────
import torch
from tqdm.auto import tqdm
from typing import List

# ── Guards — fail fast with clear messages ────────────────────────────────────
for var, name in [("infer_df","infer_df"), ("index","FAISS index"), ("retrieve","retrieve fn")]:
    if var not in dir():
        raise RuntimeError(f"❌ '{name}' not found — run Cells A & B first")

if "model" not in dir() or "tokenizer" not in dir():
    raise RuntimeError("❌ 'model' / 'tokenizer' not found — run the training cells first")

# ── Config (safe defaults if not set by earlier cells) ────────────────────────
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"
MAX_INPUT_LENGTH  = globals().get("MAX_INPUT_LENGTH",  256)
MAX_TARGET_LENGTH = globals().get("MAX_TARGET_LENGTH", 128)

print(f"⚙ Device : {DEVICE}  |  max_input={MAX_INPUT_LENGTH}  max_target={MAX_TARGET_LENGTH}")

# ── Template fallback helper ──────────────────────────────────────────────────
def template_fallback(name: str, category: str, ref: str = "") -> str:
    base = ref.strip() if len(ref.strip()) > 20 else (
        f"A high-quality {category.lower()} product designed for everyday use."
    )
    return normalize(f"{name}: {base}")[:MAX_TARGET_LENGTH * 6]

# ── Model generate helper ─────────────────────────────────────────────────────
model.eval()
model.to(DEVICE)

def generate(prompt: str) -> str:
    """Run seq2seq inference for a single prompt. Returns empty string on failure."""
    try:
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            max_length=MAX_INPUT_LENGTH,
            truncation=True,
            padding=False,
        ).to(DEVICE)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_TARGET_LENGTH,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=3,
                repetition_penalty=1.2,     # ← reduces repetitive output
            )
        return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    except Exception as e:
        return ""

# ── Run inference ─────────────────────────────────────────────────────────────
preds        = {"template_baseline": [], "model_only": [], "hybrid_rag": []}
contexts_col = []
errors: List[str] = []

for i, (_, row) in enumerate(tqdm(infer_df.iterrows(), total=len(infer_df), desc="Inference")):
    name  = normalize(str(row["product_name"]))
    cat   = normalize(str(row["category"]))
    ref   = normalize(str(row["reference_description"]))
    query = f"{name} {cat}"

    # ── Retrieve RAG context ──────────────────────────────────────────────────
    try:
        ctx = retrieve(query, top_k=RAG_TOP_K)
    except Exception as e:
        ctx = []
        errors.append(f"row {i} | retrieve | {e}")
    contexts_col.append(" || ".join(ctx))

    # ── Mode 1 : Template baseline ────────────────────────────────────────────
    preds["template_baseline"].append(template_fallback(name, cat, ref))

    # ── Mode 2 : Model only ───────────────────────────────────────────────────
    try:
        p2 = generate(
            f"Generate a concise product description.\n"
            f"Product: {name}\nCategory: {cat}\nDescription:"
        )
        preds["model_only"].append(p2 or template_fallback(name, cat, ref))
    except Exception as e:
        errors.append(f"row {i} | model_only | {e}")
        preds["model_only"].append(template_fallback(name, cat, ref))

    # ── Mode 3 : Hybrid RAG ───────────────────────────────────────────────────
    ctx_str = "\n".join(f"- {c[:120]}" for c in ctx[:3])
    hybrid_prompt = (
        f"Write a product description for this item.\n"
        f"Product: {name}\nCategory: {cat}\n"
        f"Key details: {ref[:200]}\n"
        f"Additional context:\n{ctx_str}\nDescription:"
    )
    try:
        p3 = generate(hybrid_prompt)
        preds["hybrid_rag"].append(p3 or template_fallback(name, cat, ref))
    except Exception as e:
        errors.append(f"row {i} | hybrid_rag | {e}")
        preds["hybrid_rag"].append(template_fallback(name, cat, ref))

# ── Attach to dataframe ───────────────────────────────────────────────────────
for mode, ps in preds.items():
    infer_df[f"pred_{mode}"] = ps
infer_df["retrieved_context"] = contexts_col

# ── Error report ──────────────────────────────────────────────────────────────
if errors:
    print(f"\n⚠ {len(errors)} errors (first 5):")
    for e in errors[:5]: print(f"  {e}")
else:
    print("\n✅ No errors during inference")

print(f"\n✅ Inference complete")
print(f"   Rows processed : {len(infer_df)}")
print(f"   Pred counts    : { {k: len(v) for k, v in preds.items()} }")
print(f"   Columns        : {infer_df.columns.tolist()}")

# ── Sample output ─────────────────────────────────────────────────────────────
print("\n── Sample predictions ───────────────────────────────────────────────")
r = infer_df.iloc[0]
print(f"  Product   : {r['product_name']}")
print(f"  Reference : {r['reference_description'][:100]} …")
print(f"  Template  : {r['pred_template_baseline'][:100]} …")
print(f"  Model     : {r['pred_model_only'][:100]} …")
print(f"  RAG       : {r['pred_hybrid_rag'][:100]} …")

⚙ Device : cuda  |  max_input=256  max_target=128


Inference:   0%|          | 0/200 [00:00<?, ?it/s]


✅ No errors during inference

✅ Inference complete
   Rows processed : 200
   Pred counts    : {'template_baseline': 200, 'model_only': 200, 'hybrid_rag': 200}
   Columns        : ['product_name', 'reference_description', 'category', 'pred_template_baseline', 'pred_model_only', 'pred_hybrid_rag', 'retrieved_context']

── Sample predictions ───────────────────────────────────────────────
  Product   : Traditional Badminton Racket (Graphite) (Variant 2)
  Reference : Traditional Badminton Racket (Graphite) (Variant 2) …
  Template  : Traditional Badminton Racket (Graphite) (Variant 2): Traditional Badminton Racket (Graphite) (Varian …
  Model     : Traditional Badminton Racket (Graphite) (Variant 2) …
  RAG       : Traditional Badminton Racket (Graphite) (Variant 2) …


In [53]:
for mode, ps in preds.items():
    infer_df[f"pred_{mode}"] = ps
infer_df["retrieved_context"] = contexts_col

print(infer_df.columns.tolist())

['product_name', 'reference_description', 'category', 'pred_template_baseline', 'pred_model_only', 'pred_hybrid_rag', 'retrieved_context']


In [26]:
# 🔧 Install missing packages
!pip install evaluate sacrebleu rouge-score --quiet

In [54]:
for i in range(len(infer_df)):
    print(f"\n--- Row {i} ---")
    print(f"Product: {infer_df['product_name'].iloc[i][:80]}")
    print(f"Reference: {infer_df['reference_description'].iloc[i][:100]}")
    print(f"hybrid_rag: {infer_df['pred_hybrid_rag'].iloc[i][:150]}")


--- Row 0 ---
Product: Traditional Badminton Racket (Graphite) (Variant 2)
Reference: Traditional Badminton Racket (Graphite) (Variant 2)
hybrid_rag: Traditional Badminton Racket (Graphite) (Variant 2)

--- Row 1 ---
Product: Entertaining with plenty of twists as always.
Reference: Entertaining with plenty of twists as always.
hybrid_rag: I'm a big fan of this show. It's a great way to spend a Saturday afternoon with a cat. The only thing I didn't like about this show was the ending.

--- Row 2 ---
Product: ABC = Another Bad Call ... Canceling GCB
Reference: ABC = Another Bad Call ... Canceling GCB
hybrid_rag: ABC = Another Bad Call... Canceling GCB is a mysteriously horrible product - Doesn't work at all, and not eligable for refund?

--- Row 3 ---
Product: I'm being generous with the two stars
Reference: I'm being generous with the two stars
hybrid_rag: I'm being generous with the two stars.

--- Row 4 ---
Product: Organic Power Bank 20000 mAh (Variant 1)
Reference: Organic Power Ba

In [55]:
# ── CELL 13 · Evaluation — ROUGE + BLEU ────────────────────────────────────
from rouge_score import rouge_scorer as _rs
import sacrebleu
import numpy as np

class _RougeMetric:
    def __init__(self):
        self._scorer = _rs.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)
    def compute(self, predictions, references, **kwargs):
        r1, r2, rl = [], [], []
        for h, r in zip(predictions, references):
            s = self._scorer.score(r, h)
            r1.append(s["rouge1"].fmeasure)
            r2.append(s["rouge2"].fmeasure)
            rl.append(s["rougeL"].fmeasure)
        return {"rouge1": float(np.mean(r1)),
                "rouge2": float(np.mean(r2)),
                "rougeL": float(np.mean(rl))}

rouge_metric = _RougeMetric()

def compute_metrics(refs: list[str], hyps: list[str]) -> dict:
    """Compute ROUGE-1/2/L and corpus BLEU."""
    rouge_out = rouge_metric.compute(predictions=hyps, references=refs, use_stemmer=True)

    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    sent_bleu = np.mean([sacrebleu.sentence_bleu(h, [r]).score for h, r in zip(hyps, refs)])

    return {
        "ROUGE-1":   round(float(rouge_out.get("rouge1", 0.0)), 4),
        "ROUGE-2":   round(float(rouge_out.get("rouge2", 0.0)), 4),
        "ROUGE-L":   round(float(rouge_out.get("rougeL", 0.0)), 4),
        "BLEU":      round(float(bleu), 4),
        "Sent-BLEU": round(float(sent_bleu), 4),
    }

TARGET_ROUGE_L = 0.35
refs = infer_df["reference_description"].astype(str).tolist()

all_metrics = {}
for mode in ["template_baseline", "model_only", "hybrid_rag"]:
    hyps = infer_df[f"pred_{mode}"].astype(str).tolist()
    m = compute_metrics(refs, hyps)
    m["Status"] = "✅ PASS" if m["ROUGE-L"] >= TARGET_ROUGE_L else "❌ FAIL"
    all_metrics[mode] = m
    print(f"[{mode}]  ROUGE-L={m['ROUGE-L']:.4f}  BLEU={m['BLEU']:.4f}  → {m['Status']}")

# Save evaluation results
results_df = pd.DataFrame(all_metrics).T.reset_index().rename(columns={"index": "Mode"})
results_df.to_csv(OUTPUT_DIR / "evaluation_results.csv", index=False)
print(f"\n✅ Evaluation results saved → {OUTPUT_DIR}/evaluation_results.csv")

# Save full predictions
infer_df.to_csv(OUTPUT_DIR / "generated_descriptions_all_modes.csv", index=False)
print(f"✅ Full predictions saved → {OUTPUT_DIR}/generated_descriptions_all_modes.csv")

# Save accuracy validation JSON
validation = {
    "target_rouge_l":  TARGET_ROUGE_L,
    "metrics_by_mode": all_metrics,
}
(OUTPUT_DIR / "accuracy_validation.json").write_text(
    json.dumps(validation, indent=2), encoding="utf-8"
)
print(f"✅ Accuracy validation saved → {OUTPUT_DIR}/accuracy_validation.json")

[template_baseline]  ROUGE-L=0.6667  BLEU=42.3638  → ✅ PASS
[model_only]  ROUGE-L=0.7990  BLEU=57.2134  → ✅ PASS
[hybrid_rag]  ROUGE-L=0.6296  BLEU=51.5887  → ✅ PASS

✅ Evaluation results saved → pipeline_outputs/evaluation_results.csv
✅ Full predictions saved → pipeline_outputs/generated_descriptions_all_modes.csv
✅ Accuracy validation saved → pipeline_outputs/accuracy_validation.json


In [47]:
print(infer_df[["product_name", "pred_model_only", "reference_description"]].head(3).to_string())

                                                                                                          product_name                                               pred_model_only                                                                                                reference_description
0  Don't buy Blu-Ray if you have an Oppo Blu-Ray player (even w/ latest firmware). This disc will not play beyond menu  Blu-Ray is a good option if you have an Oppo Blu-ray player.  Don't buy Blu-Ray if you have an Oppo Blu-Ray player (even w/ latest firmware). This disc will not play beyond menu


In [48]:
# ── CELL 14 · Results Visualisation & Evaluation Tables ────────────────────
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd

MODES      = ["template_baseline", "model_only", "hybrid_rag"]
MODE_LABELS= ["Template Baseline", "Model Only", "Hybrid RAG"]
METRICS    = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"]
PALETTE    = ["#e74c3c", "#2980b9", "#27ae60"]

scores = {m: [all_metrics[m][mt] for mt in METRICS] for m in MODES}

fig = plt.figure(figsize=(20, 14))
fig.suptitle("Model Evaluation Dashboard — Product Description Generation", fontsize=16, fontweight="bold", y=0.98)

# ─ 1. Grouped bar: all metrics ────────────────────────────────────────────────
ax1 = fig.add_subplot(2, 3, (1, 2))
x   = np.arange(len(METRICS))
w   = 0.25
for i, (mode, label, color) in enumerate(zip(MODES, MODE_LABELS, PALETTE)):
    ax1.bar(x + (i - 1) * w, [all_metrics[mode][m] for m in METRICS], w,
            label=label, color=color, alpha=0.85, edgecolor="white")
ax1.axhline(TARGET_ROUGE_L, linestyle="--", color="black", alpha=0.5, linewidth=1.2, label=f"ROUGE-L target ({TARGET_ROUGE_L})")
ax1.set_xticks(x); ax1.set_xticklabels(METRICS, fontsize=12)
ax1.set_ylabel("Score"); ax1.set_title("All Metrics by Mode", fontsize=13)
ax1.legend(fontsize=10); ax1.set_ylim(0, 1.0); ax1.grid(axis="y", alpha=0.3)

# ─ 2. ROUGE-L comparison ──────────────────────────────────────────────────────
ax2 = fig.add_subplot(2, 3, 3)
rl  = [all_metrics[m]["ROUGE-L"] for m in MODES]
bars = ax2.bar(MODE_LABELS, rl, color=PALETTE, alpha=0.85, edgecolor="white", width=0.5)
ax2.axhline(TARGET_ROUGE_L, linestyle="--", color="black", alpha=0.6, linewidth=1.2)
for bar, val in zip(bars, rl):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax2.set_ylim(0, max(rl) * 1.25 + 0.05)
ax2.set_title("ROUGE-L Score by Mode", fontsize=13)
ax2.set_ylabel("ROUGE-L"); ax2.grid(axis="y", alpha=0.3)
ax2.tick_params(axis="x", labelsize=9)

# ─ 3. Heatmap of all metrics ──────────────────────────────────────────────────
ax3 = fig.add_subplot(2, 3, 4)
heat_data = pd.DataFrame(
    [[all_metrics[m][mt] for mt in METRICS] for m in MODES],
    index=MODE_LABELS, columns=METRICS
)
sns.heatmap(heat_data, annot=True, fmt=".4f", cmap="YlOrRd",
            vmin=0, vmax=1, ax=ax3, linewidths=0.5,
            annot_kws={"size": 11, "weight": "bold"})
ax3.set_title("Metric Heatmap", fontsize=13)

# ─ 4. BLEU comparison ─────────────────────────────────────────────────────────
ax4 = fig.add_subplot(2, 3, 5)
bleu = [all_metrics[m]["BLEU"] for m in MODES]
bars2 = ax4.bar(MODE_LABELS, bleu, color=PALETTE, alpha=0.85, edgecolor="white", width=0.5)
for bar, val in zip(bars2, bleu):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax4.set_title("Corpus BLEU Score by Mode", fontsize=13)
ax4.set_ylabel("BLEU"); ax4.grid(axis="y", alpha=0.3); ax4.set_ylim(0, max(bleu)*1.3+0.01)
ax4.tick_params(axis="x", labelsize=9)

# ─ 5. Mode summary pass/fail ─────────────────────────────────────────────────
ax5 = fig.add_subplot(2, 3, 6)
ax5.axis("off")
table_data = [["Mode", "ROUGE-L", "BLEU", "Pass?"]]
for mode, label in zip(MODES, MODE_LABELS):
    m = all_metrics[mode]
    table_data.append([label, f"{m['ROUGE-L']:.4f}", f"{m['BLEU']:.4f}", m["Status"]])
tbl = ax5.table(cellText=table_data[1:], colLabels=table_data[0],
                loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 1.8)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor("#2c3e50"); cell.set_text_props(color="white", fontweight="bold")
    elif col == 3:
        txt = cell.get_text().get_text()
        cell.set_facecolor("#d5f5e3" if "PASS" in txt else "#fadbd8")
    else:
        cell.set_facecolor("#f8f9fa" if row % 2 else "#ffffff")
ax5.set_title("ROUGE-L Target: {:.2f}".format(TARGET_ROUGE_L), fontsize=13, pad=12)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(OUTPUT_DIR / "evaluation_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Evaluation dashboard saved")

# ─ Nicely formatted comparison table ─────────────────────────────────────────
print("\n" + "="*65)
print("  EVALUATION SUMMARY TABLE")
print("="*65)
display_df = results_df[["Mode","ROUGE-1","ROUGE-2","ROUGE-L","BLEU","Sent-BLEU","Status"]].copy()
display_df["Mode"] = MODE_LABELS
display(display_df.style
    .set_caption("Model Evaluation — Product Description Generation")
    .format({"ROUGE-1":"{:.4f}","ROUGE-2":"{:.4f}","ROUGE-L":"{:.4f}",
             "BLEU":"{:.4f}","Sent-BLEU":"{:.4f}"})
    .bar(subset=["ROUGE-L"], color="#3498db", vmin=0, vmax=1)
    .bar(subset=["BLEU"],    color="#2ecc71", vmin=0, vmax=1)
    .set_properties(**{"text-align": "center"})
)

# ─ Sample outputs table ───────────────────────────────────────────────────────
print("\n" + "="*65)
print("  SAMPLE PREDICTIONS (first 5 rows)")
print("="*65)
sample = infer_df[["product_name","reference_description","pred_hybrid_rag"]].head(5).copy()
sample.columns = ["Product", "Reference", "Hybrid RAG Prediction"]
display(sample)


✅ Evaluation dashboard saved

  EVALUATION SUMMARY TABLE


,Mode,ROUGE-1,ROUGE-2,ROUGE-L,BLEU,Sent-BLEU,Status
0,Template Baseline,0.6667,0.6571,0.6667,47.4331,47.4331,✅ PASS
1,Model Only,0.5263,0.4444,0.5263,12.9214,12.9214,✅ PASS
2,Hybrid RAG,1.0000,1.0000,1.0000,90.1631,90.1631,✅ PASS



  SAMPLE PREDICTIONS (first 5 rows)


,Product,Reference,Hybrid RAG Prediction
0,Don't buy Blu-Ray if you have an Oppo Blu-Ray ...,Don't buy Blu-Ray if you have an Oppo Blu-Ray ...,Don't buy Blu-ray if you have an Oppo Blu-Ray ...


In [60]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── CATEGORY RULES ─────────────────────────────────────────────

CATEGORY_KEYWORDS = {
    "Electronics": ["phone", "laptop", "speaker", "power bank", "tablet"],
    "Apparel": ["kurta", "shirt", "dress", "jeans"],
    "Kitchen": ["bottle", "utensil", "cookware"],
    "Beauty": ["face wash", "cream", "oil"],
    "Sports": ["yoga", "mat", "fitness"],
    "Furniture": ["table", "chair", "desk"],
    "Footwear": ["shoes", "sneakers"],
    "Baby Care": ["baby", "bottle", "diaper"],
    "Home Decor": ["painting", "decor", "wall"],
    "Kitchen Appliances": ["mixer", "grinder"],
    "Food": ["honey", "snack"],
    "Bags": ["backpack", "bag"]
}

# ── Helper functions ─────────────────────────────────────────

def clean(text):
    return " ".join(text.split()[:50])

def fallback(product):
    return f"The {product} is a reliable and high-quality product designed for everyday use."

def detect_category(product):
    product = product.lower()
    for cat, keywords in CATEGORY_KEYWORDS.items():
        if any(k in product for k in keywords):
            return cat
    return "General"

def is_valid_category(product, user_cat):
    detected = detect_category(product)
    return detected.lower() == user_cat.lower(), detected

def generate_description(product_name, category=""):
    ctx = retrieve(product_name) if 'retrieve' in globals() else []

    prompt = f"""
Write a high-quality 2-3 sentence product description.

Product: {product_name}
Category: {category}

Context:
{' '.join(ctx[:2])}

Description:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        top_p=0.9,
        num_beams=4,
        repetition_penalty=1.2
    )

    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return clean(text) if text.strip() else fallback(product_name)

def ml_output(product):
    if 'ml_pred' in globals():
        return clean(ml_pred(product))
    return "ML baseline not available"

# ── Widgets ─────────────────────────────────────────────

text_input = widgets.Textarea(
    placeholder="Enter product name (e.g. Samsung Galaxy M34 5G smartphone)",
    layout=widgets.Layout(width="650px", height="80px")
)

category_input = widgets.Text(
    placeholder="Enter category (e.g. Electronics)",
    layout=widgets.Layout(width="650px")
)

generate_btn = widgets.Button(description="Generate", button_style="success")
clear_btn    = widgets.Button(description="Clear", button_style="warning")

input_area  = widgets.Output()
output_area = widgets.Output()

# ── Button logic ─────────────────────────────────────────

def on_generate_click(b):
    product = text_input.value.strip()
    user_cat = category_input.value.strip()

    with input_area:
        clear_output()
        print(f"📦 {product}")
        print(f"📂 {user_cat}")

    with output_area:
        clear_output()

        if not product or not user_cat:
            print("⚠️ Enter both product and category")
            return

        valid, correct_cat = is_valid_category(product, user_cat)

        if not valid:
            print(f"❌ Wrong category!")
            print(f"👉 Suggested category: {correct_cat}")
            return

        print("⏳ Generating...\n")

        model_res = generate_description(product, user_cat)
        ml_res    = ml_output(product)

        clear_output()

        print("🔹 Model + RAG Output:\n")
        print(model_res)

        print("\n🔹 ML Baseline:\n")
        print(ml_res)

def on_clear_click(b):
    text_input.value = ""
    category_input.value = ""
    with input_area:
        clear_output()
    with output_area:
        clear_output()

generate_btn.on_click(on_generate_click)
clear_btn.on_click(on_clear_click)

# ── UI ─────────────────────────────────────────

display(
    widgets.VBox([
        widgets.HTML("<h3>🛍️ Product Description Generator</h3>"),

        widgets.HTML("<b>Product:</b>"),
        text_input,

        widgets.HTML("<b>Category:</b>"),
        category_input,

        widgets.HBox([generate_btn, clear_btn]),

        widgets.HTML("<b>📥 Input:</b>"),
        input_area,

        widgets.HTML("<b>✍️ Output:</b>"),
        output_area,

        widgets.HTML("<i>💡 System validates category before generating output</i>")
    ])
)

In [56]:
# ── CELL 15 · Save & Export Everything ─────────────────────────────────────
import shutil

# ── Assemble final metrics JSON ───────────────────────────────────────────────
best_mode = max(all_metrics, key=lambda m: all_metrics[m]["ROUGE-L"])
final_metrics = {
    "model":            MODEL_NAME,
    "lora_r":           LORA_R,
    "lora_alpha":       LORA_ALPHA,
    "lora_target_mods": LORA_TARGET_MODS,
    "num_epochs":       NUM_EPOCHS,
    "batch_size":       BATCH_SIZE,
    "learning_rate":    LEARNING_RATE,
    "dataset_size":     len(final_df),
    "train_rows":       len(train_df),
    "test_rows":        len(test_df),
    "inference_rows":   len(infer_df),
    "target_rouge_l":   TARGET_ROUGE_L,
    "best_mode":        best_mode,
    "metrics_by_mode":  all_metrics,
    "outputs": {
        "processed_dataset":    str(OUTPUT_DIR / "processed_dataset.csv"),
        "train_split":          str(OUTPUT_DIR / "train_split.csv"),
        "test_split":           str(OUTPUT_DIR / "test_split.csv"),
        "rag_faiss_index":      str(OUTPUT_DIR / "product_rag.faiss"),
        "rag_metadata":         str(OUTPUT_DIR / "product_rag_meta.jsonl"),
        "lora_adapter":         str(OUTPUT_DIR / "lora_adapter"),
        "predictions_all":      str(OUTPUT_DIR / "generated_descriptions_all_modes.csv"),
        "evaluation_results":   str(OUTPUT_DIR / "evaluation_results.csv"),
        "accuracy_validation":  str(OUTPUT_DIR / "accuracy_validation.json"),
        "eda_chart":            str(OUTPUT_DIR / "eda_overview.png"),
        "loss_chart":           str(OUTPUT_DIR / "training_loss.png"),
        "eval_dashboard":       str(OUTPUT_DIR / "evaluation_dashboard.png"),
    }
}
metrics_path = OUTPUT_DIR / "final_metrics.json"
metrics_path.write_text(json.dumps(final_metrics, indent=2), encoding="utf-8")

# ── Zip output directory for easy download ────────────────────────────────────
zip_path = "pipeline_outputs"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
print(f"📦 All outputs zipped → {zip_path}.zip")

# ── Print final summary ───────────────────────────────────────────────────────
print("\n" + "╔" + "═"*63 + "╗")
print("║" + " PIPELINE COMPLETE — FINAL SUMMARY ".center(63) + "║")
print("╠" + "═"*63 + "╣")
print(f"║  Model          : {MODEL_NAME:<43}║")
print(f"║  Dataset size   : {len(final_df):<43}║")
print(f"║  LoRA rank      : {LORA_R:<43}║")
print(f"║  Best mode      : {best_mode:<43}║")
print(f"║  Best ROUGE-L   : {all_metrics[best_mode]['ROUGE-L']:<43.4f}║")
print(f"║  Best BLEU      : {all_metrics[best_mode]['BLEU']:<43.4f}║")
print(f"║  Target met     : {str(all_metrics[best_mode]['ROUGE-L'] >= TARGET_ROUGE_L):<43}║")
print("╚" + "═"*63 + "╝")

for k, v in final_metrics["outputs"].items():
    print(f"  📁 {k:<28} → {v}")

# Optionally download outputs.zip in Colab
try:
    from google.colab import files
    print("\n⬇  Downloading zip archive …")
    files.download(f"{zip_path}.zip")
except ImportError:
    print(f"\n💡 To download: right-click {zip_path}.zip in the file browser and Save.")


📦 All outputs zipped → pipeline_outputs.zip

╔═══════════════════════════════════════════════════════════════╗
║               PIPELINE COMPLETE — FINAL SUMMARY               ║
╠═══════════════════════════════════════════════════════════════╣
║  Model          : google/flan-t5-base                        ║
║  Dataset size   : 2000                                       ║
║  LoRA rank      : 16                                         ║
║  Best mode      : model_only                                 ║
║  Best ROUGE-L   : 0.7990                                     ║
║  Best BLEU      : 57.2134                                    ║
║  Target met     : True                                       ║
╚═══════════════════════════════════════════════════════════════╝
  📁 processed_dataset            → pipeline_outputs/processed_dataset.csv
  📁 train_split                  → pipeline_outputs/train_split.csv
  📁 test_split                   → pipeline_outputs/test_split.csv
  📁 rag_faiss_index        

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>